# 🚗 Domácí úkol (Session 2): Neuronové sítě – Regrese spotřeby paliva (Auto MPG)
**Dataset:** `auto_mpg.csv` (398 záznamů automobilů z let 1970–1982)  
**Cíl úlohy:** Návrh, trénování a vyhodnocení plně propojené dopředné neuronové sítě (**Multilayer Perceptron – MLP**) v knihovně **Keras / TensorFlow** pro predikci spotřeby paliva v mílích na galon (**MPG – Miles Per Gallon**).

Zdroj dat: [UCI Machine Learning Repository – Auto MPG Dataset](https://archive.ics.uci.edu/dataset/9/auto+mpg) (původně StatLib Carnegie Mellon University).

---
### 📋 Kroky zadání:
1. **Načtení dat:** Pomocí knihovny Pandas načtěte dataset `auto_mpg.csv`.
2. **Datové profilování:** Pomocí `ydata_profiling` vygenerujte profilovací report, ověřte chybějící hodnoty, korelace a distribuce.
3. **Odstranění nepotřebných proměnných:** Vyřaďte nestrukturovaný sloupec `car_name`.
4. **Transformace proměnné origin:** Překódujte číselné hodnoty podle klíče: `1 -> 'USA'`, `2 -> 'Europe'`, `3 -> 'Japan'`.
5. **Ošetření neplatných hodnot:** V proměnné `horsepower` nahraďte znaky `'?'` hodnotou `NaN` a odstraňte tyto řádky.
6. **Kódování a normalizace:** Převeďte kategorické proměnné pomocí `.get_dummies()` a proveďte normalizaci nezávislých proměnných pomocí `StandardScaler`.
7. **Rozdělení dat:** Rozdělte data na trénovací a testovací množinu (`train_test_split`).
8. **Stavba neuronové sítě:** Vytvořte model v knihovně Keras (`Sequential`, `Dense`).
9. **Kompilace:** Nastavte optimalizátor **Adam** a ztrátovou funkci **MSE** (Mean Squared Error).
10. **Trénování:** Natrénujte síť s vhodným počtem epoch, velikostí dávky (`batch_size`) a validačním poměrem (`validation_split`).
11. **Predikce & Vyhodnocení:** Proveďte odhady na testovací množině `X_test` a spočítejte metriky **MAE** a **RMSE**.


In [ ]:
# Krok 1: Import knihoven a načtení datasetu
import os
os.environ['KERAS_BACKEND'] = 'torch'  # Podpora Keras 3 na PyTorch backendu

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import keras
from keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Načtení dat s flexibilním vyhledáním cesty
csv_candidates = [
    'auto_mpg.csv',
    'data/auto_mpg.csv',
    'data/MAL_downloadable materials_session 2/Homework/auto_mpg.csv'
]
csv_path = next(p for p in csv_candidates if os.path.exists(p))
auto_df = pd.read_csv(csv_path)

# Úprava názvů sloupců (odstranění mezer)
auto_df.columns = auto_df.columns.str.strip().str.replace(' ', '_')

print(f"Úspěšně načten dataset Auto MPG! Tvar: {auto_df.shape}")
print(f"Názvy sloupců: {list(auto_df.columns)}")
auto_df.head(5)


### 📊 Krok 2: Profilování dat pomocí ydata_profiling
Vygenerujeme automatizovaný profilovací report s analýzou chybějících hodnot, korelací a distribucí.

In [ ]:
# Krok 2: Generování profilovacího reportu (ydata_profiling)
try:
    import ydata_profiling as ydp
    report = ydp.ProfileReport(auto_df, title="Auto MPG Profiling Report", minimal=True)
    report.to_file("auto_mpg_report.html")
    print("Profilovací report úspěšně vygenerován do auto_mpg_report.html!")
except ImportError:
    print("Knihovna ydata_profiling není nainstalována, provádíme profilovací kontrolu přes Pandas:")

# Rychlá inspekce klíčových zjištění
print("\n1. Datové typy a nenulové počty:")
print(auto_df.dtypes)
print("\n2. Znaky '?' v proměnné horsepower:", (auto_df['horsepower'] == '?').sum())
print("\n3. Rozdělení proměnné origin:")
print(auto_df['origin'].value_counts())


### 🧹 Krok 3, 4 & 5: Čištění dat a Transformace proměnných
1. **Odstranění `car_name`:** Název vozidla je volný text s vysokou kardinalitou a chybami v zápisu, který nepřináší přímou fyzikální informaci o spotřebě.
2. **Mapování `origin`:** `1 -> 'USA'`, `2 -> 'Europe'`, `3 -> 'Japan'`.
3. **Ošetření neplatných hodnot:** 6 řádků obsahuje `'?'` v poli `horsepower`. Nahradíme je `np.nan` a tyto řádky odstraníme (zůstane 392 záznamů).

In [ ]:
# Krok 3: Odstranění nepotřebné proměnné car_name
clean_df = auto_df.drop(columns=['car_name']).copy()

# Krok 4: Transformace proměnné origin
origin_map = {1: 'USA', 2: 'Europe', 3: 'Japan'}
clean_df['origin'] = clean_df['origin'].map(origin_map)

# Krok 5: Nahrazení neplatných hodnot '?' hodnotou NaN a odstranění řádků
clean_df['horsepower'] = pd.to_numeric(clean_df['horsepower'].replace('?', np.nan))
rows_before = len(clean_df)
clean_df = clean_df.dropna().reset_index(drop=True)
print(f"Odstraněno {rows_before - len(clean_df)} neplatných řádků. Čistých vzorků: {len(clean_df)}")

# Krok 6: One-Hot kódování kategorické proměnné origin pomocí get_dummies
df_encoded = pd.get_dummies(clean_df, columns=['origin'], drop_first=True, dtype=int)
print("Výsledné příznaky:", list(df_encoded.columns))
df_encoded.head(3)


### ⚖️ Krok 6 & 7: Rozdělení dat a Normalizace (StandardScaler)
Data rozdělíme na trénovací sadu (80 %) a testovací sadu (20 %) s fixovaným `random_state=42`.  
Nezávislé proměnné $X$ normalizujeme pomocí `StandardScaler` (z-score: nulový průměr, jednotkový rozptyl), aby gradientní optimalizace Kerasu konvergovala hladce bez numerické nestability.

In [ ]:
# Krok 7: Rozdělení na trénovací a testovací množinu
feature_cols = [c for c in df_encoded.columns if c != 'mpg']
X = df_encoded[feature_cols].values
y = df_encoded['mpg'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Normalizace nezávislých proměnných (fit na train, transform na train i test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Trénovací sada: {X_train_scaled.shape[0]} vzorků, {X_train_scaled.shape[1]} příznaků")
print(f"Testovací sada : {X_test_scaled.shape[0]} vzorků")


### 🧠 Krok 8 & 9: Stavba a kompilace neuronové sítě Keras
Vytvoříme dopřednou neuronovou síť (MLP):
- **Vstupní vrstva:** 8 příznaků (`cylinders`, `displacement`, `horsepower`, `weight`, `acceleration`, `model_year`, `origin_Japan`, `origin_USA`).
- **Skrytá vrstva 1:** 64 neuronů, aktivace **ReLU**.
- **Skrytá vrstva 2:** 32 neuronů, aktivace **ReLU**.
- **Výstupní vrstva:** 1 neuron (lineární aktivace pro spojitou regresi MPG).
- **Kompilace:** Optimalizátor **Adam** ($lr = 0.01$), ztrátová funkce **MSE** (Mean Squared Error), metrika **MAE**.

In [ ]:
# Krok 8 & 9: Definice a kompilace Keras modelu
keras.utils.set_random_seed(42)

model = models.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu', name="dense_1"),
    layers.Dense(32, activation='relu', name="dense_2"),
    layers.Dense(1, name="output_mpg")
], name="Auto_MPG_Regressor")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='mse',
    metrics=['mae']
)

model.summary()


### 🚀 Krok 10: Trénování neuronové sítě
Trénujeme na trénovací sadě s 20% validačním dělením (`validation_split=0.20`), dávkou velikosti 16 vzorků (`batch_size=16`) a po dobu 100 epoch.

In [ ]:
# Krok 10: Trénování modelu
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=16,
    verbose=1
)

# Vykreslení křivek učení (Ztrátová funkce MSE a MAE)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Trénovací MSE', color='#2563eb', lw=2)
axes[0].plot(history.history['val_loss'], label='Validační MSE', color='#dc2626', lw=2, linestyle='--')
axes[0].set_title('Vývoj ztrátové funkce (MSE Loss)')
axes[0].set_xlabel('Epocha')
axes[0].set_ylabel('Mean Squared Error')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Trénovací MAE', color='#059669', lw=2)
axes[1].plot(history.history['val_mae'], label='Validační MAE', color='#d97706', lw=2, linestyle='--')
axes[1].set_title('Vývoj metriky MAE (Průměrná absolutní chyba)')
axes[1].set_xlabel('Epocha')
axes[1].set_ylabel('MAE [MPG]')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 🎯 Krok 11, 12 & 13: Predikce na testovací sadě & Výpočet MAE a RMSE

In [ ]:
# Krok 11 & 12: Predikce na X_test
y_pred = model.predict(X_test_scaled, verbose=0).flatten()

# Krok 13: Výpočet MAE, MSE a RMSE
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("           VÝSLEDKY NA TESTOVACÍ SADĚ")
print("=" * 50)
print(f"Průměrná absolutní chyba (MAE)  : {mae:.3f} MPG")
print(f"Odmocnina stř. čtverce (RMSE)   : {rmse:.3f} MPG")
print(f"Střední čtvercová chyba (MSE)   : {mse:.3f} MPG²")
print(f"Koeficient determinace (R²)     : {r2:.4f}")
print("=" * 50)

# Vykreslení Skutečnost vs. Predikce
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.75, color='#2563eb', edgecolors='k', s=50, label='Testovací automobily')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideální predikce (y = ŷ)')
plt.title(f'Skutečná vs. Predikovaná spotřeba MPG (R² = {r2:.3f})')
plt.xlabel('Skutečná spotřeba [MPG]')
plt.ylabel('Predikovaná spotřeba neuronovou sítí [MPG]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### 🇪🇺 Bonus: Přepočet na evropské jednotky (L/100 km)
Vztah pro přepočet mezi americkými mílemi na galon (MPG) a evropskou spotřebou v litrech na 100 kilometrů je nepřímo úměrný:
$$\text{Spotřeba [L/100 km]} = \frac{235.215}{\text{MPG}}$$

In [ ]:
# Ukázka srovnání prvních 10 vozidel z testovací sady
sample_df = pd.DataFrame({
    'Skutečné MPG': np.round(y_test[:10], 1),
    'Predikované MPG': np.round(y_pred[:10], 1),
    'Chyba |y - ŷ| [MPG]': np.round(np.abs(y_test[:10] - y_pred[:10]), 2),
    'Skutečná spotřeba [L/100km]': np.round(235.215 / y_test[:10], 2),
    'Predikce [L/100km]': np.round(235.215 / y_pred[:10], 2)
})
sample_df
